# FLAN-T5 Construction Rules Training on Google Colab

This notebook fine-tunes a FLAN-T5 model to convert natural language construction requirements (ASHRAE 90.1-2013) into structured JSON, then evaluates the result against baselines.

**Hardware**: Works on GPU or CPU. A GPU (T4 or better) is recommended — CPU-only training works but is much slower.

**Estimated training time** (5 epochs, ~2750 examples):
- CPU only: several hours
- T4 GPU: ~45-60 minutes
- L4 GPU: ~25-35 minutes
- A100 GPU: ~15-20 minutes

## Step 1: Check GPU/CPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected - training will run on CPU and will be much slower.")
    print("Optional: Runtime > Change runtime type > Hardware accelerator > GPU")

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers>=4.30.0 datasets>=2.0.0 accelerate>=0.20.0 sentencepiece>=0.1.99 pandas matplotlib seaborn openai
print("Dependencies installed.")

## Step 3: Upload Dataset

**Option A**: Upload the dataset file directly (recommended for Colab)

In [ ]:
from google.colab import files
import os

print("Please upload your dataset file: construction_ashrae_2013.jsonl")
uploaded = files.upload()

dataset_path = list(uploaded.keys())[0]
print(f"Dataset uploaded: {dataset_path}")
print(f"File size: {os.path.getsize(dataset_path) / 1024 / 1024:.2f} MB")

**Option B**: Clone from GitHub instead of uploading manually

In [ ]:
# Uncomment and modify if you'd rather clone the repo and use the dataset from it
# !git clone https://github.com/YOUR_USERNAME/DL-Construction-Recommendation.git
# dataset_path = "DL-Construction-Recommendation/dataset/construction_ashrae_2013.jsonl"

## Step 4: Define Training Functions

Mirrors `src/train_t5.py` in the repo so results here are reproducible outside Colab too.

In [ ]:
import json
from typing import Dict, List, Any
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset


def load_jsonl_dataset(file_path: str) -> List[Dict[str, Any]]:
    """Load a JSONL dataset file."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data


def prepare_dataset(jsonl_path: str, tokenizer, max_input_length: int = 512, max_target_length: int = 512):
    """Tokenize the dataset; dynamic padding is left to the DataCollator."""
    data = load_jsonl_dataset(jsonl_path)

    inputs = [item["input_text"] for item in data]
    targets = [item["target_json"] for item in data]

    dataset = Dataset.from_dict({"input_text": inputs, "target_json": targets})

    def tokenize_function(examples):
        model_inputs = tokenizer(
            examples["input_text"],
            max_length=max_input_length,
            truncation=True,
            padding=False
        )
        labels = tokenizer(
            text_target=examples["target_json"],
            max_length=max_target_length,
            truncation=True,
            padding=False
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)


print("Training functions defined.")

## Step 5: Configure Training Parameters

In [ ]:
CONFIG = {
    "model_name": "google/flan-t5-base",  # or "google/flan-t5-small" for faster/CPU-friendly training
    "output_dir": "./flan_t5_construction",
    "learning_rate": 5e-5,
    "batch_size": 8,  # reduce to 2-4 if training on CPU or if you hit OOM
    "num_epochs": 5,
    "weight_decay": 0.01,
    "max_input_length": 256,
    "max_target_length": 512,
    "eval_split": 0.1
}

print("Training configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Step 6: Load Model and Tokenizer

In [ ]:
print(f"Loading model: {CONFIG['model_name']}...")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

print("Model loaded.")
print(f"Model parameters: {model.num_parameters() / 1e6:.1f}M")

## Step 7: Prepare Dataset

In [ ]:
print(f"Loading dataset from {dataset_path}...")
dataset = prepare_dataset(
    dataset_path,
    tokenizer,
    CONFIG["max_input_length"],
    CONFIG["max_target_length"]
)

dataset = dataset.train_test_split(test_size=CONFIG["eval_split"], seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")

## Step 8: Setup Training Arguments

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

steps_per_epoch = max(1, len(train_dataset) // CONFIG["batch_size"])
eval_save_steps = max(50, steps_per_epoch // 3)

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Evaluation/Save every {eval_save_steps} steps")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["num_epochs"],
    weight_decay=CONFIG["weight_decay"],
    logging_dir=f"{CONFIG['output_dir']}/logs",
    logging_steps=50,
    save_steps=eval_save_steps,
    eval_steps=eval_save_steps,
    eval_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    push_to_hub=False,
    report_to="none",
    fp16=torch.cuda.is_available(),  # mixed precision only makes sense on GPU
)

print("Training arguments configured.")

## Step 9: Initialize Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer initialized.")

## Step 10: Start Training

In [ ]:
import time

print("=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print(f"Model: {CONFIG['model_name']}")
print(f"Device: {device}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print("=" * 80)

start_time = time.time()
trainer.train()
training_time = (time.time() - start_time) / 60

print("\n" + "=" * 80)
print(f"TRAINING COMPLETE in {training_time:.2f} minutes")
print("=" * 80)

## Step 11: Visualize Training Metrics

In [ ]:
import matplotlib.pyplot as plt

def plot_training_metrics(trainer, save_path="training_metrics.png"):
    """Plot training/eval loss curves from the trainer's log history."""
    log_history = trainer.state.log_history

    train_logs = [log for log in log_history if "loss" in log and "eval_loss" not in log]
    eval_logs = [log for log in log_history if "eval_loss" in log]

    if not train_logs or not eval_logs:
        print("Not enough logged history to plot yet.")
        return

    train_steps = [log["step"] for log in train_logs]
    train_loss = [log["loss"] for log in train_logs]
    eval_steps = [log["step"] for log in eval_logs]
    eval_loss = [log["eval_loss"] for log in eval_logs]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_steps, train_loss, label="Training Loss", linewidth=2)
    ax.plot(eval_steps, eval_loss, label="Validation Loss", linewidth=2, marker="o", markersize=4)
    ax.set_xlabel("Training Steps")
    ax.set_ylabel("Loss")
    ax.set_title("FLAN-T5 Fine-tuning Loss")
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Initial training loss: {train_loss[0]:.4f}  |  Final: {train_loss[-1]:.4f}")
    print(f"Best validation loss:  {min(eval_loss):.4f}")


plot_training_metrics(trainer)

## Step 12: Save Final Model

In [ ]:
print(f"Saving model to {CONFIG['output_dir']}...")
trainer.save_model()
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Model saved.")

## Step 13: Evaluate Model

In [ ]:
print("Running final evaluation...")
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## Step 14: Test Inference

Quick sanity check with automatic JSON repair for the common case where the model drops a nested `{`.

In [ ]:
import re

def repair_json(text: str) -> str:
    """Best-effort repair for common T5-generated JSON issues: missing outer
    braces, missing braces around nested objects (inputs/outputs/units/notes),
    unquoted keys, and trailing commas."""
    text = text.strip()
    text = re.sub(r'^```json\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^```\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'```\s*$', '', text, flags=re.IGNORECASE)

    if not text.startswith("{"):
        text = "{" + text
    if not text.endswith("}"):
        text = text + "}"

    # Insert missing "{" after nested-object keys, and close that object right
    # before the next known top-level key. Only fires when the brace is
    # actually missing, so well-formed JSON passes through unchanged.
    NEST_ORDER = ("inputs", "outputs", "units", "notes")
    for i, key in enumerate(NEST_ORDER):
        text, n_open = re.subn(rf'"{key}"\s*:\s*"', f'"{key}": {{"', text)
        if not n_open:
            continue
        next_key = NEST_ORDER[i + 1] if i + 1 < len(NEST_ORDER) else None
        if next_key:
            text = re.sub(rf',\s*"{next_key}":', f'}}, "{next_key}":', text)
        else:
            # "notes" is the last nested object - close it right before the final outer closing brace
            text = text[:-1].rstrip() + "}" + "}"

    text = re.sub(r',\s*}', '}', text)
    text = re.sub(r',\s*]', ']', text)
    return text


test_input = "In climate zone 5, exterior walls of type SteelFramed must not exceed a U-factor of 0.064."
print(f"Input: {test_input}")

inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=512, num_beams=5, do_sample=False, early_stopping=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nRaw generated output:")
print(generated_text)

parsed = None
try:
    parsed = json.loads(generated_text)
    print("\nValid JSON on first try.")
except json.JSONDecodeError:
    fixed = repair_json(generated_text)
    try:
        parsed = json.loads(fixed)
        print("\nRepaired invalid JSON successfully.")
    except json.JSONDecodeError as e:
        print(f"\nCould not repair JSON: {e}")

if parsed:
    print(json.dumps(parsed, indent=2))

## Step 15: Download Trained Model

In [ ]:
zip_path = f"{CONFIG['output_dir'].strip('./')}.zip"
!zip -rq {zip_path} {CONFIG['output_dir']}

print(f"Model zipped: {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.2f} MB)")

files.download(zip_path)
print("Download started - check your browser's download folder.")

## Step 16: Numeric Field-Level Evaluation

Format-agnostic comparison: extracts numeric values (U-value, SHGC, F-factor, C-factor, VT) from the raw model output via regex and compares them against the dataset's ground truth, so the metric doesn't depend on getting the JSON braces perfectly right.

In [ ]:
import random
import pandas as pd

FIELDS = ["max_u_value", "max_shgc", "max_f_factor", "max_c_factor", "min_vt"]
FIELD_LABELS = {
    "max_u_value": "U-value",
    "max_shgc": "SHGC",
    "max_f_factor": "F-factor",
    "max_c_factor": "C-factor",
    "min_vt": "VT",
}
FIELD_PATTERNS = {
    "max_u_value": [r'"max_u_value"\s*:\s*([\d.]+)', r'U-?(?:value|factor)["\s:=]*([\d.]+)', r'U\s*[≤<=]\s*([\d.]+)'],
    "max_shgc": [r'"max_shgc"\s*:\s*([\d.]+)', r'SHGC["\s:=]*([\d.]+)', r'SHGC\s*[≤<=]\s*([\d.]+)'],
    "max_f_factor": [r'"max_f_factor"\s*:\s*([\d.]+)', r'F-?factor["\s:=]*([\d.]+)'],
    "max_c_factor": [r'"max_c_factor"\s*:\s*([\d.]+)', r'C-?factor["\s:=]*([\d.]+)'],
    "min_vt": [r'"min_vt"\s*:\s*([\d.]+)', r'\bVT\s*[≥>=]*\s*([\d.]+)'],
}


def extract_numeric_values(text: str) -> Dict[str, float]:
    """Extract known numeric fields from raw model output, any formatting."""
    results = {}
    for field, patterns in FIELD_PATTERNS.items():
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                try:
                    results[field] = float(match.group(1))
                    break
                except ValueError:
                    continue
    return results


def extract_ground_truth(target_json) -> Dict[str, Any]:
    if isinstance(target_json, str):
        target_json = json.loads(target_json)
    outputs = target_json.get("outputs", {})
    return {field: outputs.get(field) for field in FIELDS}


def compare_numeric_values(prediction_text: str, ground_truth: Dict[str, Any], tolerance: float = 0.001) -> Dict[str, Any]:
    predicted = extract_numeric_values(prediction_text)
    all_fields = [f for f, v in ground_truth.items() if v is not None]
    correct_fields = [
        f for f in all_fields
        if predicted.get(f) is not None and abs(predicted[f] - ground_truth[f]) < tolerance
    ]
    if not all_fields:
        return {"correct": False, "accuracy": 0.0, "details": {}}
    details = {f: {"ground_truth": ground_truth[f], "prediction": predicted.get(f), "correct": f in correct_fields} for f in all_fields}
    return {
        "correct": len(correct_fields) == len(all_fields),
        "accuracy": len(correct_fields) / len(all_fields),
        "details": details,
    }


def generate_prediction(model, tokenizer, input_text: str) -> str:
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=512, num_beams=5, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print("Running numeric field-level evaluation on a sample of the dataset...")

raw_data = load_jsonl_dataset(dataset_path)
random.seed(42)
eval_samples = random.sample(raw_data, k=min(30, len(raw_data)))

field_eval_results = []
for ex in eval_samples:
    gt = extract_ground_truth(ex["target_json"])
    if all(v is None for v in gt.values()):
        continue
    pred_text = generate_prediction(model, tokenizer, ex["input_text"])
    comparison = compare_numeric_values(pred_text, gt)
    field_eval_results.append({
        "input": ex["input_text"],
        "ground_truth": gt,
        "accuracy": comparison["accuracy"],
        "correct": comparison["correct"],
    })

df_field_eval = pd.DataFrame(field_eval_results)
print(f"\nExamples evaluated: {len(df_field_eval)}")
print(f"Field-level accuracy:   {df_field_eval['accuracy'].mean():.1%}")
print(f"Example-level accuracy: {df_field_eval['correct'].mean():.1%}")

## Step 17: Baseline Comparison

Compares the fine-tuned model against the original (non-fine-tuned) FLAN-T5, a naive constant-value baseline, and optionally ChatGPT (only if an `OPENAI_API_KEY` secret or environment variable is available — this step is skipped otherwise).

To enable the ChatGPT comparison in Colab: click the key icon in the left sidebar ("Secrets"), add a secret named `OPENAI_API_KEY`, and grant this notebook access to it. No API key should ever be hardcoded in this file.

In [ ]:
def get_openai_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENAI_API_KEY")


OPENAI_API_KEY = get_openai_api_key()
if OPENAI_API_KEY:
    print("OPENAI_API_KEY found - ChatGPT baseline will be included.")
else:
    print("No OPENAI_API_KEY secret/env var set - ChatGPT baseline will be skipped (optional).")


def load_original_flan_model(model_name="google/flan-t5-base", device=None):
    print(f"Loading original (non-fine-tuned) {model_name}...")
    original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    original_tokenizer = AutoTokenizer.from_pretrained(model_name)
    if device is not None:
        original_model = original_model.to(device)
    original_model.eval()
    return original_model, original_tokenizer


def naive_predict(input_text: str) -> str:
    """Always predicts the dataset's most common max_u_value as a floor baseline."""
    return '"max_u_value": 0.06'


def chatgpt_predict(input_text: str, api_key: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=200,
        messages=[
            {"role": "system", "content": (
                "You are an expert at extracting numeric building code requirements. "
                "Extract U-values, F-factors, C-factors, SHGC, and VT from the text. "
                "Return the values as: max_u_value: X, max_shgc: Y, max_f_factor: Z, max_c_factor: W, min_vt: V"
            )},
            {"role": "user", "content": f"Extract numeric requirements from: {input_text}"},
        ],
    )
    return response.choices[0].message.content


def run_baseline_comparison(model, tokenizer, dataset_path, num_samples=50, openai_api_key=None):
    print("=" * 80)
    print("BASELINE COMPARISON")
    print("=" * 80)

    device = model.device
    original_model, original_tokenizer = load_original_flan_model(device=device)

    raw_data = load_jsonl_dataset(dataset_path)
    valid_samples = [ex for ex in raw_data if any(v is not None for v in extract_ground_truth(ex["target_json"]).values())]

    random.seed(42)
    eval_samples = random.sample(valid_samples, k=min(num_samples, len(valid_samples)))
    print(f"Evaluating on {len(eval_samples)} examples")

    rows = []
    for i, ex in enumerate(eval_samples, 1):
        print(f"Processing {i}/{len(eval_samples)}...", end="\r")
        gt = extract_ground_truth(ex["target_json"])

        finetuned_out = generate_prediction(model, tokenizer, ex["input_text"])
        finetuned_cmp = compare_numeric_values(finetuned_out, gt)

        original_out = generate_prediction(original_model, original_tokenizer, ex["input_text"])
        original_cmp = compare_numeric_values(original_out, gt)

        naive_out = naive_predict(ex["input_text"])
        naive_cmp = compare_numeric_values(naive_out, gt)

        chatgpt_cmp = None
        if openai_api_key:
            try:
                chatgpt_out = chatgpt_predict(ex["input_text"], openai_api_key)
                chatgpt_cmp = compare_numeric_values(chatgpt_out, gt)
            except Exception as e:
                print(f"\nChatGPT call failed: {e}")

        rows.append({
            "id": f"Q{i:03d}",
            "input": ex["input_text"],
            "finetuned_accuracy": finetuned_cmp["accuracy"],
            "finetuned_correct": finetuned_cmp["correct"],
            "finetuned_details": finetuned_cmp["details"],
            "original_accuracy": original_cmp["accuracy"],
            "original_correct": original_cmp["correct"],
            "original_details": original_cmp["details"],
            "naive_accuracy": naive_cmp["accuracy"],
            "naive_correct": naive_cmp["correct"],
            "chatgpt_accuracy": chatgpt_cmp["accuracy"] if chatgpt_cmp else None,
            "chatgpt_correct": chatgpt_cmp["correct"] if chatgpt_cmp else None,
        })

    print(" " * 40)
    df = pd.DataFrame(rows)

    print(f"\n{'Model':<28} {'Field Accuracy':<18} {'Example Accuracy':<18}")
    print("-" * 64)
    print(f"{'FLAN-T5 (fine-tuned)':<28} {df['finetuned_accuracy'].mean():>16.1%} {df['finetuned_correct'].mean():>17.1%}")
    print(f"{'FLAN-T5 (original)':<28} {df['original_accuracy'].mean():>16.1%} {df['original_correct'].mean():>17.1%}")
    if df["chatgpt_accuracy"].notna().any():
        print(f"{'ChatGPT (zero-shot)':<28} {df['chatgpt_accuracy'].mean():>16.1%} {df['chatgpt_correct'].mean():>17.1%}")
    print(f"{'Naive baseline':<28} {df['naive_accuracy'].mean():>16.1%} {df['naive_correct'].mean():>17.1%}")

    return df


df_comparison = run_baseline_comparison(model, tokenizer, dataset_path, num_samples=50, openai_api_key=OPENAI_API_KEY)
df_comparison.to_csv("baseline_comparison_results.csv", index=False)
print("\nResults saved to baseline_comparison_results.csv")

## Step 18: Comparison Figures

In [ ]:
import numpy as np
import seaborn as sns

plt.rcParams["figure.dpi"] = 150

def plot_model_comparison(df, save_path="model_comparison.png"):
    models, field_acc, example_acc, colors = [], [], [], []

    models.append("FLAN-T5\n(fine-tuned)"); field_acc.append(df["finetuned_accuracy"].mean() * 100); example_acc.append(df["finetuned_correct"].mean() * 100); colors.append("#FF6B6B")
    models.append("FLAN-T5\n(original)"); field_acc.append(df["original_accuracy"].mean() * 100); example_acc.append(df["original_correct"].mean() * 100); colors.append("#95E1D3")
    if df["chatgpt_accuracy"].notna().any():
        models.append("ChatGPT\n(zero-shot)"); field_acc.append(df["chatgpt_accuracy"].mean() * 100); example_acc.append(df["chatgpt_correct"].mean() * 100); colors.append("#4ECDC4")
    models.append("Naive\nbaseline"); field_acc.append(df["naive_accuracy"].mean() * 100); example_acc.append(df["naive_correct"].mean() * 100); colors.append("#C7C7C7")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    x = np.arange(len(models))

    for ax, values, title in [(axes[0], field_acc, "Field-Level Accuracy"), (axes[1], example_acc, "Example-Level Accuracy")]:
        bars = ax.bar(x, values, color=colors, alpha=0.85, edgecolor="black")
        ax.set_xticks(x)
        ax.set_xticklabels(models, fontsize=9)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(title)
        ax.grid(True, alpha=0.3, linestyle="--", axis="y")
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 1, f"{h:.1f}%", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_per_field_heatmap(df, save_path="field_accuracy_heatmap.png"):
    def field_accuracy(details_col, field):
        correct = total = 0
        for details in df[details_col]:
            if isinstance(details, dict) and field in details:
                total += 1
                if details[field]["correct"]:
                    correct += 1
        return (correct / total * 100) if total else np.nan

    rows_data, row_names = [], []
    rows_data.append([field_accuracy("finetuned_details", f) for f in FIELDS]); row_names.append("Fine-tuned")
    rows_data.append([field_accuracy("original_details", f) for f in FIELDS]); row_names.append("Original")

    df_heat = pd.DataFrame(rows_data, columns=[FIELD_LABELS[f] for f in FIELDS], index=row_names)
    fig, ax = plt.subplots(figsize=(7, max(3, len(row_names) * 1.2)))
    sns.heatmap(df_heat, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, ax=ax, cbar_kws={"label": "Accuracy (%)"})
    ax.set_title("Per-Field Accuracy by Model")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_model_comparison(df_comparison)
plot_per_field_heatmap(df_comparison)

## Done

- Model weights: `CONFIG['output_dir']` (zipped and downloaded in Step 15)
- Loss curve: `training_metrics.png`
- Comparison figures: `model_comparison.png`, `field_accuracy_heatmap.png`
- Raw comparison data: `baseline_comparison_results.csv`

To run the same pipeline outside Colab, use `src/train_t5.py` and `src/run_inference.py` from the repo — they implement the same training/inference logic without the Colab-specific upload/download/secrets steps.